# Crop Insurance Sales Dashboard
USDA RMA Summary of Business · the newest crop year (partial, early season) and the 3 before it.
Green = premium · red = indemnity · blue = liability · violet = policies sold · orange = net acres.

In [ ]:
# --- Settings ---------------------------------------------------------------------
# Setup notes, the refresh rules and caveats are in README.md.
REFRESH_ON_OPEN   = True   # check RMA for new data every time the dashboard opens (~1s when nothing changed)
PIPELINE_YEARS    = None   # e.g. [2025, 2026]; None = newest crop year + HISTORY_YEARS prior
HISTORY_YEARS     = 3      # prior years to load alongside the newest (partial) crop year
CHECK_PRIOR_YEARS = False  # True = also re-check prior years on RMA (revised as late losses come in)
TOP_N             = 15     # bars shown in the ranking charts

import logging
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as w
from IPython.display import display, HTML, clear_output

import data_pipeline as dp

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")  # keep the view free of log noise
logging.getLogger("urllib3").setLevel(logging.ERROR)  # no retry chatter when RMA is unreachable
pd.options.display.float_format = "{:,.0f}".format

In [ ]:
# --- Load data (refreshing from RMA first) --------------------------------------
from datetime import datetime

refresh_error = None
if REFRESH_ON_OPEN:
    try:   # only the newest crop year is checked; unchanged data is reused, so this is quick
        dp.run_pipeline(years=PIPELINE_YEARS, history=HISTORY_YEARS, check_all=CHECK_PRIOR_YEARS)
    except Exception as exc:
        refresh_error = exc
df, meta = dp.load_processed()
PARTIAL = set(meta.get("partial_years", []))

def year_label(y) -> str:
    return f"{int(y)} (partial)" if int(y) in PARTIAL else str(int(y))

def _local(ts: str) -> str:
    return datetime.fromisoformat(ts).astimezone().strftime("%b %d, %Y %I:%M %p")

status = (f'<span style="color:#a15c00">Couldn&#39;t reach RMA just now ({type(refresh_error).__name__}); '
          f'showing data last checked {_local(meta.get("checked_at", meta["built_at"]))}.</span>'
          if refresh_error else f'Up to date with RMA as of {_local(meta.get("checked_at", meta["built_at"]))}.')
display(HTML(f'<div style="font:13px system-ui,-apple-system,Segoe UI,sans-serif;color:#52514e;margin:4px 0">{status}</div>'))

In [ ]:
# --- Design tokens -------------------------------------------------------------
FONT    = 'system-ui, -apple-system, "Segoe UI", Roboto, sans-serif'
INK     = "#0b0b0b"   # primary text
INK_2   = "#52514e"   # secondary text
MUTED   = "#898781"   # axis labels, captions
GRID    = "#e1e0d9"   # hairline gridlines
AXIS    = "#c3c2b7"   # baselines
SURFACE = "#fcfcfb"   # chart background
NEUTRAL = "#898781"   # measures with no color of their own

# One color per MEASURE, used everywhere that measure appears: the stat-card accent, bars,
# the state-map ramp and the table header. Validated for colorblind separation where two
# appear together (premium vs indemnity), with a legend and bar gaps as secondary cues.
METRICS = {  # column -> (label, color, sequential ramp for the map)
    "total_prem":       ("Total Premium ($)",        "#008300", "Greens"),
    "indem_amt":        ("Indemnity ($)",            "#e34948", "Reds"),
    "liability_amt":    ("Liability ($)",            "#2a78d6", "Blues"),
    "pol_sold_cnt":     ("Policies Sold",            "#4a3aa7", "Purples"),
    "net_acre_qty":     ("Net Acres",                "#eb6834", "Oranges"),
    "subsidy":          ("Subsidy ($)",              NEUTRAL,   "Greys"),
    "farmer_paid_prem": ("Farmer-paid Premium ($)",  NEUTRAL,   "Greys"),
    "pol_prem_cnt":     ("Policies Earning Premium", NEUTRAL,   "Greys"),
}
LABEL = {k: v[0] for k, v in METRICS.items()}
COLOR = {k: v[1] for k, v in METRICS.items()}
RAMP  = {k: v[2] for k, v in METRICS.items()}
DOLLAR_METRICS = {"total_prem", "liability_amt", "subsidy", "farmer_paid_prem", "indem_amt"}
CHART_METRICS = [("Premium", "total_prem"), ("Indemnity", "indem_amt"), ("Liability", "liability_amt"),
                 ("Policies Sold", "pol_sold_cnt"), ("Net Acres", "net_acre_qty")]

# --- Plotly theme: quiet chrome so the data carries the color ------------------
_axis = dict(gridcolor=GRID, linecolor=AXIS, zerolinecolor=AXIS, showline=False, ticks="",
             tickfont=dict(color=MUTED, size=11), title=dict(font=dict(color=MUTED, size=11)))
_layout = dict(
    font=dict(family=FONT, size=12, color=INK_2),
    title=dict(font=dict(size=15, color=INK), x=0, xanchor="left"),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    xaxis=_axis, yaxis=_axis,
    hoverlabel=dict(bgcolor="white", bordercolor=GRID, font=dict(family=FONT, size=12, color=INK)),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1, title=dict(text=""),
                font=dict(color=INK_2)),
    margin=dict(l=8, r=8, t=56, b=8),
    bargap=0.28, bargroupgap=0.06,
)
pio.templates["sob"] = go.layout.Template(layout=_layout)
try:
    pio.templates["sob"].layout.barcornerradius = 3   # rounded bar ends (plotly >= 5.19)
except (ValueError, AttributeError):
    pass
pio.templates.default = "plotly_white+sob"
PLOT_CONFIG = {"displaylogo": False, "modeBarButtonsToRemove": ["lasso2d", "select2d", "autoScale2d"]}

# --- Aggregation & formatting --------------------------------------------------
def summarize(frame: pd.DataFrame, by=None) -> pd.DataFrame:
    """Sum the metrics (optionally grouped) and add ratios computed AFTER aggregation."""
    cols = list(METRICS)
    agg = frame[cols].sum().to_frame().T if by is None else frame.groupby(by, observed=True)[cols].sum().reset_index()
    prem = agg["total_prem"].where(agg["total_prem"] != 0)
    agg["loss_ratio"] = agg["indem_amt"] / prem
    agg["subsidy_pct"] = agg["subsidy"] / prem
    agg["prem_rate"] = agg["total_prem"] / agg["liability_amt"].where(agg["liability_amt"] != 0)
    return agg

def fmt_money(v) -> str:
    """1 decimal with B/M/K suffix, e.g. $12.3M."""
    for div, suf in ((1e9, "B"), (1e6, "M"), (1e3, "K")):
        if abs(v) >= div:
            return f"${v/div:,.1f}{suf}"
    return f"${v:,.1f}"

def fmt_value(metric, v) -> str:
    return fmt_money(v) if metric in DOLLAR_METRICS else f"{v:,.0f}"

def axis_values(series: pd.Series, metric: str):
    """Chart values: dollars in $ millions (1 decimal), counts as-is."""
    if metric in DOLLAR_METRICS:
        return (series / 1e6).round(1), LABEL[metric].replace("($)", "($M)")
    return series, LABEL[metric]

def short(label: str, n: int = 26) -> str:
    return label if len(label) <= n else label[: n - 1] + "…"

# --- Stat cards ------------------------------------------------------------------
def kpi_cards(frame: pd.DataFrame) -> str:
    s = summarize(frame).iloc[0]
    lr = f"{s.loss_ratio:.2f}" if pd.notna(s.loss_ratio) else "–"
    sub_pct = f"{s.subsidy_pct:.1%} of premium" if pd.notna(s.subsidy_pct) else ""
    cards = [  # (label, value, caption, accent color or None)
        ("Total Premium", fmt_money(s.total_prem), f"{fmt_money(s.farmer_paid_prem)} farmer-paid", COLOR["total_prem"]),
        ("Indemnity", fmt_money(s.indem_amt), "losses paid", COLOR["indem_amt"]),
        ("Loss Ratio", lr, "indemnity ÷ premium", None),
        ("Liability", fmt_money(s.liability_amt), "coverage in force", COLOR["liability_amt"]),
        ("Policies Sold", f"{s.pol_sold_cnt:,.0f}", f"{s.pol_prem_cnt:,.0f} earning premium", COLOR["pol_sold_cnt"]),
        ("Net Acres", f"{s.net_acre_qty:,.0f}", "reported × share", COLOR["net_acre_qty"]),
        ("Subsidy", fmt_money(s.subsidy), sub_pct, None),
    ]
    tiles = "".join(
        f'<div style="flex:1 1 150px;background:{SURFACE};border:1px solid rgba(11,11,11,.10);border-radius:10px;'
        f'padding:12px 14px 12px;border-top:3px solid {c or "rgba(11,11,11,.10)"}">'
        f'<div style="font-size:12px;color:{INK_2};letter-spacing:.02em">{k}</div>'
        f'<div style="font-size:22px;font-weight:600;color:{INK};margin-top:2px">{v}</div>'
        f'<div style="font-size:11px;color:{MUTED};margin-top:2px">{cap}</div></div>'
        for k, v, cap, c in cards)
    return f'<div style="display:flex;flex-wrap:wrap;gap:10px;margin:8px 0 14px;font-family:{FONT}">{tiles}</div>' 

In [ ]:
ALL = "(All)"

def options(col, frame=df):
    vals = sorted(frame[col].dropna().astype(str).unique().tolist())
    if col == "crop_yr":  # newest first; partial year labelled, value stays the plain year
        return [ALL] + [(year_label(v), v) for v in sorted(vals, reverse=True)]
    return [ALL] + vals

def _values(opts):
    return [o[1] if isinstance(o, tuple) else o for o in opts]

def _dropdown(col):
    return w.Dropdown(options=options(col), layout=w.Layout(width="220px"))

def _labeled(text, widget):
    return w.VBox([w.HTML(f'<span style="font:12px {FONT};color:{INK_2}">{text}</span>'), widget],
                  layout=w.Layout(margin="0 12px 0 0"))

f_year, f_comm, f_plan, f_state = (_dropdown(c) for c in ("crop_yr", "commodity", "insurance_plan", "state_abbr"))
f_metric = w.ToggleButtons(options=CHART_METRICS, style={"button_width": "auto"})
b_reset = w.Button(description="Reset", icon="refresh", layout=w.Layout(width="90px", margin="18px 0 0 0"))

FILTERS = {"crop_yr": f_year, "commodity": f_comm, "insurance_plan": f_plan, "state_abbr": f_state}
FILTER_NAMES = {"crop_yr": "Crop year", "commodity": "Commodity", "insurance_plan": "Plan", "state_abbr": "State"}

def filtered(exclude=None) -> pd.DataFrame:
    """Apply every active filter (optionally skipping one, used to build cascading options)."""
    mask = pd.Series(True, index=df.index)
    for col, widget in FILTERS.items():
        if col != exclude and widget.value != ALL:
            mask &= df[col].astype(str) == widget.value
    return df[mask]

_updating = False
def refresh_options():
    """Cascade: each dropdown only lists values that exist under the *other* active filters."""
    global _updating
    _updating = True
    try:
        for col, widget in FILTERS.items():
            current = widget.value
            opts = options(col, filtered(exclude=col))
            widget.options = opts
            widget.value = current if current in _values(opts) else ALL
    finally:
        _updating = False

In [ ]:
import re

PLAN_DESCRIPTIONS = {
    "YP":     "Yield Protection: pays when the farm's yield falls below its guaranteed yield.",
    "RP":     "Revenue Protection: pays when farm revenue drops below the guarantee; guarantee rises if the harvest price goes up.",
    "RPHPE":  "Revenue Protection with Harvest Price Exclusion: like RP, but the guarantee doesn't rise with the harvest price (cheaper).",
    "APH":    "Actual Production History: yield coverage based on the farm's own yield history (common for specialty and perennial crops).",
    "ARH":    "Actual Revenue History: revenue coverage based on the farm's own revenue history (specialty crops).",
    "AYP":    "Area Yield Protection: pays when the county's average yield drops, not the individual farm's.",
    "ARP":    "Area Revenue Protection: pays when county-level revenue drops below the guarantee.",
    "ARPHPE": "Area Revenue Protection with Harvest Price Exclusion: county revenue coverage without the harvest-price increase.",
    "SCO":    "Supplemental Coverage Option: county-based add-on that covers part of the underlying policy's deductible.",
    "ECO":    "Enhanced Coverage Option: county-based add-on covering a shallow loss band near the top of coverage.",
    "STAX":   "Stacked Income Protection: county-based revenue coverage for upland cotton.",
    "MP":     "Margin Protection: county-based coverage against a drop in revenue minus input costs.",
    "RI":     "Rainfall Index: pays when rainfall in a grid area falls below normal (pasture, rangeland, forage).",
    "PRF":    "Pasture, Rangeland, Forage (Rainfall Index): pays when grid-area rainfall falls below normal.",
    "VI":     "Vegetation Index: pays when a satellite greenness index for the area falls below normal.",
    "WFRP":   "Whole-Farm Revenue Protection: covers the revenue of the entire farm under one policy.",
    "MFRP":   "Micro Farm Revenue Protection: simplified whole-farm revenue coverage for small farms.",
    "DO":     "Dollar plan: covers a fixed dollar amount per acre when crop value falls below it.",
    "DOL":    "Dollar plan: covers a fixed dollar amount per acre when crop value falls below it.",
    "PACE":   "Post-Application Coverage Endorsement: covers yield loss when planned in-season nitrogen can't be applied.",
    "HIPWI":  "Hurricane Insurance Protection - Wind Index: pays when a named hurricane's winds hit the county.",
    "LRP":    "Livestock Risk Protection: protects against a decline in livestock prices.",
    "LGM":    "Livestock Gross Margin: protects the margin between livestock prices and feed costs.",
    "DRP":    "Dairy Revenue Protection: protects against a drop in quarterly milk revenue.",
    "GRP":    "Group Risk Plan (legacy): county-yield-based coverage, replaced by Area Yield Protection.",
    "GRIP":   "Group Risk Income Protection (legacy): county-revenue-based coverage, replaced by Area Revenue Protection.",
}

def plan_description(plan_label: str):
    """'RPHPE (03)' -> description. Exact abbreviation first, then prefix match (e.g. SCO-RP, ECO-YP)."""
    abbr = re.sub(r"[^A-Z]", "", plan_label.split("(")[0].upper())
    if abbr in PLAN_DESCRIPTIONS:
        return PLAN_DESCRIPTIONS[abbr]
    for key in sorted(PLAN_DESCRIPTIONS, key=len, reverse=True):
        if abbr.startswith(key) and len(key) >= 2:
            return PLAN_DESCRIPTIONS[key]
    return None

In [ ]:
def _left_titles(fig, starts):
    """Left-align subplot titles over their panels (plotly centers them by default)."""
    for ann, x in zip(fig.layout.annotations, starts):
        ann.update(x=x, xanchor="left", font=dict(size=15, color=INK))

def _col_starts(widths, spacing):
    total = 1 - spacing * (len(widths) - 1)
    starts, x = [], 0.0
    for wdt in widths:
        starts.append(x)
        x += total * wdt / sum(widths) + spacing
    return starts

def year_chart(frame):
    """Premium vs indemnity by crop year on ONE $M axis; loss ratio as a label above each year."""
    agg = summarize(frame, "crop_yr").sort_values("crop_yr")
    years = [year_label(y) for y in agg["crop_yr"]]
    fig = go.Figure()
    for m in ("total_prem", "indem_amt"):
        vals, _ = axis_values(agg[m], m)
        fig.add_bar(x=years, y=vals, name=LABEL[m].replace(" ($)", ""), marker_color=COLOR[m],
                    hovertemplate="%{x}<br>" + LABEL[m].replace(" ($)", "") + ": $%{y:,.1f}M<extra></extra>")
    top = (agg[["total_prem", "indem_amt"]].max(axis=1) / 1e6).round(1)
    for x, y, lr in zip(years, top, agg["loss_ratio"]):
        if pd.notna(lr):
            fig.add_annotation(x=x, y=y, text=f"Loss ratio <b>{lr:.2f}</b>", showarrow=False, yshift=14,
                               font=dict(size=12, color=INK_2))
    fig.update_layout(title="Premium, Indemnity & Loss Ratio by Crop Year", height=380, barmode="group",
                      yaxis=dict(title=dict(text="$ millions"), rangemode="tozero"),
                      xaxis=dict(showgrid=False, tickfont=dict(color=INK_2, size=12)))
    fig.update_yaxes(range=[0, float(top.max()) * 1.15 if len(top) and top.max() > 0 else 1])
    return fig

def ranking_pair(frame, metric):
    """Top commodities and top insurance plans, side by side, in the metric's color."""
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.2,
                        subplot_titles=(f"Top {TOP_N} Commodities by {LABEL[metric].replace(' ($)', '')}",
                                        f"Top {TOP_N} Insurance Plans by {LABEL[metric].replace(' ($)', '')}"))
    n_max = 1
    for col, by in enumerate(("commodity", "insurance_plan"), start=1):
        agg = summarize(frame, by).nlargest(TOP_N, metric).sort_values(metric)
        vals, axis_title = axis_values(agg[metric], metric)
        n_max = max(n_max, len(agg))
        fmt = "$%{x:,.1f}M" if metric in DOLLAR_METRICS else "%{x:,.0f}"
        fig.add_trace(go.Bar(
            x=vals, y=[short(v) for v in agg[by]], orientation="h", marker_color=COLOR[metric],
            customdata=list(zip(agg[by], agg["loss_ratio"].fillna(0))), showlegend=False,
            hovertemplate="<b>%{customdata[0]}</b><br>" + fmt + "<br>Loss ratio %{customdata[1]:.2f}<extra></extra>"),
            row=1, col=col)
        fig.update_xaxes(title_text=axis_title, row=1, col=col)
        fig.update_yaxes(showgrid=False, tickfont=dict(color=INK_2, size=11), row=1, col=col)
    _left_titles(fig, _col_starts([1, 1], 0.2))
    fig.update_layout(height=max(380, 24 * n_max + 110), bargap=0.3)
    return fig

def map_and_coverage(frame, metric):
    """State map (single-hue ramp of the metric's color) beside the coverage-level distribution."""
    widths, spacing = [0.58, 0.42], 0.06
    label = LABEL[metric].replace(" ($)", "")
    fig = make_subplots(rows=1, cols=2, column_widths=widths, horizontal_spacing=spacing,
                        specs=[[{"type": "choropleth"}, {"type": "xy"}]],
                        subplot_titles=(f"{label} by State", f"{label} by Coverage Level"))
    st = summarize(frame, "state_abbr")
    z, axis_title = axis_values(st[metric], metric)
    fmt = "$%{z:,.1f}M" if metric in DOLLAR_METRICS else "%{z:,.0f}"
    fig.add_trace(go.Choropleth(
        locations=st["state_abbr"], z=z, locationmode="USA-states", colorscale=RAMP[metric],
        marker_line_color="white", marker_line_width=0.6, customdata=st["loss_ratio"].fillna(0),
        colorbar=dict(title=dict(text=axis_title, side="top", font=dict(size=11, color=MUTED)),
                      orientation="h", x=widths[0] * (1 - spacing) / 2, xanchor="center",
                      y=-0.02, yanchor="top", len=0.4, thickness=10, outlinewidth=0,
                      tickfont=dict(size=10, color=MUTED)),
        hovertemplate="<b>%{location}</b><br>" + fmt + "<br>Loss ratio %{customdata:.2f}<extra></extra>"),
        row=1, col=1)
    fig.update_geos(scope="usa", bgcolor=SURFACE, lakecolor=SURFACE, showlakes=False,
                    landcolor="#f0efec", subunitcolor="white")

    cov = summarize(frame, "cov_lvl_pct")
    cov = cov[(cov[metric] > 0) & (cov["cov_lvl_pct"] > 0)].sort_values("cov_lvl_pct")
    vals, _ = axis_values(cov[metric], metric)
    yfmt = "$%{y:,.1f}M" if metric in DOLLAR_METRICS else "%{y:,.0f}"
    fig.add_trace(go.Bar(
        x=[f"{int(c)}%" for c in cov["cov_lvl_pct"]], y=vals, marker_color=COLOR[metric], showlegend=False,
        hovertemplate="Coverage %{x}<br>" + yfmt + "<extra></extra>"), row=1, col=2)
    fig.update_xaxes(title_text="Coverage level", type="category", showgrid=False, row=1, col=2)
    fig.update_yaxes(title_text=axis_title, row=1, col=2)
    _left_titles(fig, _col_starts(widths, spacing))
    fig.update_layout(height=430, margin=dict(l=8, r=8, t=56, b=40))
    return fig

def summary_table(frame):
    """Top products (commodity x plan) by premium. $ columns in millions, 1 decimal."""
    t = summarize(frame, ["commodity", "insurance_plan"]).sort_values("total_prem", ascending=False).head(25)
    names = {}
    for col in METRICS:
        label = LABEL[col]
        if col in DOLLAR_METRICS:
            t[col] = t[col] / 1e6
            label = label.replace("($)", "($M)")
        names[col] = label
    t = t.rename(columns={**names, "commodity": "Commodity", "insurance_plan": "Plan",
                          "loss_ratio": "Loss Ratio", "subsidy_pct": "Subsidy %", "prem_rate": "Premium Rate"})
    fmts = {names[c]: ("{:,.1f}" if c in DOLLAR_METRICS else "{:,.0f}") for c in METRICS}
    accent = [{"selector": f"th.col_heading.col{list(t.columns).index(names[c])}",
               "props": [("border-top", f"3px solid {COLOR[c]}")]}
              for c in ("total_prem", "indem_amt", "liability_amt", "pol_sold_cnt", "net_acre_qty")]
    return (t.style.format(fmts)
             .format({"Loss Ratio": "{:.2f}", "Subsidy %": "{:.1%}", "Premium Rate": "{:.1%}"}, na_rep="–")
             .hide(axis="index")
             .set_table_styles([
                 {"selector": "", "props": [("border-collapse", "collapse"), ("font-family", FONT), ("font-size", "12px")]},
                 {"selector": "th", "props": [("background", "#f6f5f2"), ("color", INK), ("font-weight", "600"),
                                              ("padding", "6px 10px"), ("text-align", "right"),
                                              ("border-bottom", f"1px solid {AXIS}")]},
                 {"selector": "td", "props": [("padding", "5px 10px"), ("text-align", "right"),
                                              ("border-bottom", f"1px solid {GRID}"),
                                              ("font-variant-numeric", "tabular-nums")]},
                 {"selector": "th.col_heading.col0, th.col_heading.col1, td.col0, td.col1",
                  "props": [("text-align", "left")]},
                 {"selector": "tbody tr:hover", "props": [("background", "#f6f5f2")]},
                 *accent]))

In [ ]:
out = w.Output()

def _heading(sub) -> str:
    active = [f"{FILTER_NAMES[c]}: <b>{year_label(wd.value) if c == 'crop_yr' else wd.value}</b>"
              for c, wd in FILTERS.items() if wd.value != ALL]
    title = " · ".join(active) if active else "All sales data combined"
    years = ", ".join(year_label(y) for y in sorted(sub["crop_yr"].unique()))
    return (f'<div style="font-family:{FONT};margin:6px 0 2px">'
            f'<div style="font-size:18px;font-weight:600;color:{INK}">{title}</div>'
            f'<div style="font-size:12px;color:{MUTED}">Crop years {years} · {len(sub):,} records</div></div>')

def download_link(sub) -> str:
    """CSV of the current view (year x commodity x plan), embedded as a browser download link."""
    import base64
    csv = summarize(sub, ["crop_yr", "commodity", "insurance_plan"]).to_csv(index=False)
    href = "data:text/csv;base64," + base64.b64encode(csv.encode()).decode()
    name = f"crop_insurance_view_{datetime.now():%Y%m%d}.csv"
    return (f'<a download="{name}" href="{href}" style="font:13px {FONT};color:{COLOR["liability_amt"]}">'
            f'Download this view as CSV</a>')

def render(*_):
    if _updating:
        return
    refresh_options()
    sub = filtered()
    metric = f_metric.value
    with out:
        clear_output(wait=True)
        display(HTML(_heading(sub)))
        if sub.empty:
            display(HTML(f'<p style="font-family:{FONT};color:{INK_2}">No records match these filters.</p>'))
            return
        if f_plan.value != ALL:
            desc = plan_description(f_plan.value) or "No description on file for this plan (add one to PLAN_DESCRIPTIONS)."
            display(HTML(f'<div style="font:13px {FONT};color:{INK_2};font-style:italic;margin:2px 0 6px">{desc}</div>'))
        display(HTML(kpi_cards(sub)))
        year_chart(sub).show(config=PLOT_CONFIG)          # row 1: trend
        ranking_pair(sub, metric).show(config=PLOT_CONFIG)  # row 2: commodities | plans
        map_and_coverage(sub, metric).show(config=PLOT_CONFIG)  # row 3: state map | coverage level
        display(HTML(f'<div style="font:600 15px {FONT};color:{INK};margin:18px 0 6px">'
                     f'Top Products (Commodity × Plan) by Premium</div>'))
        display(summary_table(sub))                        # row 4: table
        display(HTML(download_link(sub)))

def reset(_):
    global _updating
    _updating = True
    for widget in FILTERS.values():
        widget.value = ALL
    _updating = False
    render()

for widget in [*FILTERS.values(), f_metric]:
    widget.observe(render, names="value")
b_reset.on_click(reset)

controls = w.VBox([
    w.HBox([_labeled("Crop year", f_year), _labeled("Commodity", f_comm),
            _labeled("Insurance plan", f_plan), _labeled("State", f_state), b_reset],
           layout=w.Layout(flex_flow="row wrap", align_items="flex-start")),
    w.HBox([w.HTML(f'<span style="font:12px {FONT};color:{INK_2};margin-right:8px">Chart metric</span>'), f_metric],
           layout=w.Layout(align_items="center", margin="8px 0 0 0")),
], layout=w.Layout(padding="10px 12px", border=f"1px solid {GRID}", margin="0 0 6px 0"))
display(controls, out)
render()  # default view: everything combined